## Задание 1. Разработка и тестирование определения тональности текста

### Шаг 1. Загрузка и подготовка данных

In [34]:
from pathlib import Path
from datasets import load_dataset
import pandas as pd


DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

SPLITS = [
    "train",
    "test",
    "validation"
]


def save_data(dataset):
    for split_name, split in dataset.items():
        if not (DATA_DIR / f"{split_name}.csv").exists():
            split.to_csv(DATA_DIR / f"{split_name}.csv", index=False)
            print(split_name, len(split))


if all((DATA_DIR / f"{t}.csv").exists() for t in SPLITS):
    print("Данные уже загружены")
else:
    ds = load_dataset("jbeno/sentiment_merged")
    save_data(ds)

train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")
validation_df = pd.read_csv(DATA_DIR / "validation.csv")

df_columns = train_df.columns
head = train_df.head()

print(df_columns)
print(head)

for split, df in zip(SPLITS, [train_df, test_df, validation_df]):
    print('-' * 50)
    print(f"{split.upper()}")
    print(f"Shape: {df.shape}")

    print("\nРаспределение классов:")
    print(df["label"].value_counts(normalize=True))

    print("\nПропуски:")
    print(df.isna().sum())
    print()



Данные уже загружены
Index(['sentence', 'label', 'source', 'split'], dtype='str')
                                            sentence     label       source  \
0  Those 2 drinks are part of the HK culture and ...  negative  dynasent_r2   
1  I was told by the repair company that was doin...  negative  dynasent_r1   
2             It is there to give them a good time .   neutral    sst_local   
3  Like leafing through an album of photos accomp...  negative    sst_local   
4         Johnny was a talker and liked to have fun.  positive  dynasent_r1   

   split  
0  train  
1  train  
2  train  
3  train  
4  train  
--------------------------------------------------
TRAIN
Shape: (102097, 4)

Распределение классов:
label
neutral     0.481385
positive    0.304015
negative    0.214600
Name: proportion, dtype: float64

Пропуски:
sentence    0
label       0
source      0
split       0
dtype: int64

--------------------------------------------------
TEST
Shape: (6530, 4)

Распределение классо

### Шаг 2. Предобработка текстов

In [35]:
import nltk
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

for res in ["punkt", "punkt_tab", "stopwords", "wordnet", "averaged_perceptron_tagger_eng"]:
    nltk.download(res, quiet=True)

1. Токенизация (разделение на слова).
2. Приведение к нижнему регистру.
3. Удаление стоп-слов.
4. Лемматизация (нормализация слов)

In [36]:
NEGATIONS = {"no", "not", "nor", "never"}
STOP_WORDS = set(stopwords.words("english")) - NEGATIONS

lemmatizer = WordNetLemmatizer()


def to_wordnet_pos(tag):
    if tag.startswith("J"):
        return wordnet.ADJ
    if tag.startswith("V"):
        return wordnet.VERB
    if tag.startswith("R"):
        return wordnet.ADV
    return wordnet.NOUN


def preprocess(text):
    tokens = word_tokenize(str(text).lower())

    # n't -> not, чтобы отрицание пережило фильтр isalpha
    tokens = ["not" if t == "n't" else t for t in tokens]

    # части речи определяем ДО удаления слов: теггеру нужен контекст
    tagged = nltk.pos_tag(tokens)

    result = []
    for word, tag in tagged:
        # 3. только слова из букв (выкидывает пунктуацию, числа, кавычки)
        if not word.isalpha():
            continue
        # 4. стоп-слова
        if word in STOP_WORDS:
            continue
        # 5. лемматизация с учётом части речи
        result.append(lemmatizer.lemmatize(word, to_wordnet_pos(tag)))

    # 6. строка для TfidfVectorizer
    return " ".join(result)


for text in train_df["sentence"].head(10):
    print(text)
    print("->", preprocess(text))
    print("-" * 50)


CLEAN_PATH = {s: DATA_DIR / f"{s}_clean.parquet" for s in SPLITS}
dfs = {"train": train_df, "test": test_df, "validation": validation_df}

for name, df in dfs.items():
    if CLEAN_PATH[name].exists():
        dfs[name] = pd.read_parquet(CLEAN_PATH[name])
    else:
        df["clean_text"] = df["sentence"].apply(preprocess)
        df.to_parquet(CLEAN_PATH[name], index=False)
    print(name, "пустых после очистки:", (dfs[name]["clean_text"] == "").sum())

train_df, test_df, validation_df = dfs["train"], dfs["test"], dfs["validation"]

Those 2 drinks are part of the HK culture and has years of history. It is so bad.
-> drink part hk culture year history bad
--------------------------------------------------
I was told by the repair company that was doing the car repair that fixing the rim was "impossible" and to replace it.
-> tell repair company car repair fix rim impossible replace
--------------------------------------------------
It is there to give them a good time .
-> give good time
--------------------------------------------------
Like leafing through an album of photos accompanied by the sketchiest of captions .
-> like leaf album photo accompany sketchiest caption
--------------------------------------------------
Johnny was a talker and liked to have fun.
-> johnny talker like fun
--------------------------------------------------
It as burnt to a crisp black flavorless
-> burnt crisp black flavorless
--------------------------------------------------
I called Moveaholics Jason was amazing he even offered

### Шаг 3. Преобразование текстов в векторы
- Bag-of-Words или TF-IDF.
- Для лексиконного метода — вычислите «тональный балл» по словарю.

In [37]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()

# Применение TF-IDF к текстовым данным
tfidf_matrix = vectorizer.fit_transform(train_df["clean_text"])

# Получение списка ключевых слов и их значения TF-IDF для первого документа
feature_names = vectorizer.get_feature_names_out()
tfidf_scores = tfidf_matrix.toarray()[0]

# Сортировка слов по значениям TF-IDF
sorted_keywords = [word for _, word in sorted(zip(tfidf_scores, feature_names), reverse=True)]

print("Ключевые слова:", sorted_keywords)


Ключевые слова: ['hk', 'culture', 'history', 'part', 'drink', 'bad', 'year', '非常正宗的湖南菜', '难得有像中国味的烤串', '美式日料', '念の為反対でもトライした', '居心地最高コスパもいいですよ是非ラスベガスに来たら寄ってみてくださいね', '厨师推荐', '分子虎', 'わかってはいたけど', 'über', 'únicas', 'être', 'événements', 'évité', 'évitez', 'éviter', 'évidemment', 'été', 'étrange', 'étoiles', 'étions', 'étape', 'étant', 'était', 'étaient', 'étagères', 'étage', 'établissement', 'équilibrés', 'épaisse', 'énormément', 'élevé', 'élever', 'élan', 'également', 'écoute', 'éclipse', 'échantillon', 'ça', 'área', 'ángeles', 'zéro', 'zzzzzzzzz', 'zzzzzzz', 'zzzz', 'zz', 'zwick', 'zwei', 'zwar', 'zuzu', 'zurich', 'zur', 'zuppa', 'zupas', 'zuni', 'zumba', 'zum', 'zukes', 'zugabe', 'zufrieden', 'zucker', 'zucchini', 'zu', 'zoom', 'zoolander', 'zoo', 'zong', 'zone', 'zombie', 'zoes', 'zoe', 'znoud', 'ziyi', 'ziva', 'zitti', 'ziti', 'zishe', 'zippys', 'zippy', 'zipps', 'zippo', 'zipper', 'zip', 'zion', 'zinnis', 'zings', 'zingernation', 'zinger', 'zing', 'zinc', 'zinburger', 'zina', 'zin',